# BBO Capstone — Round 5 Query Submission
**Imperial College London · Executive Master in ML/AI**  
**Gian Franco Cattaneo · COO, SALOV S.p.A.**  
**Week 16 — Required Capstone Component 16.1: Refining the BBO Strategy**

---

## Architecture notes
- **GP surrogate**: Matérn-5/2 ARD kernel + WhiteKernel, `StandardScaler` on X, `n_restarts_optimizer=10`
- **Acquisition**: Expected Improvement — **maximisation convention** (corrected from R4)
- **NN surrogate**: MLPRegressor 2×32 ReLU, lbfgs solver, α=1e-3 — used for gradient/sensitivity analysis
- **Data**: R1–R4 portal inputs and outputs (4 observations per function)
- **Objective**: maximisation of unknown scalar function across 8 black-box functions (d=2–8)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel, WhiteKernel
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor

from scipy.stats import norm
from scipy.optimize import minimize

np.random.seed(42)
print('All libraries loaded successfully.')

In [ ]:
# ── FULL DATASET: Rounds 1–4 ─────────────────────────────────────────────────

R1_X = [
    np.array([0.034388, 0.909319]),
    np.array([0.695196, 0.395970]),
    np.array([0.548145, 0.174647, 0.303245]),
    np.array([0.440429, 0.425456, 0.378357, 0.397088]),
    np.array([0.000000, 0.675974, 0.999999, 0.999999]),
    np.array([0.464677, 0.242110, 0.574863, 0.999999, 0.000000]),
    np.array([0.000000, 0.241713, 0.327655, 0.218095, 0.375335, 0.747501]),
    np.array([0.064016, 0.008062, 0.123268, 0.000000, 0.999999, 0.381742, 0.031402, 0.806010])
]
R2_X = [
    np.array([0.999999, 0.999999]),
    np.array([0.698486, 0.000000]),
    np.array([0.850892, 0.035316, 0.936193]),
    np.array([0.999999, 0.000000, 0.000000, 0.365908]),
    np.array([0.000000, 0.000000, 0.999999, 0.999999]),
    np.array([0.142733, 0.321812, 0.416485, 0.999999, 0.304415]),
    np.array([0.000000, 0.302741, 0.000000, 0.187177, 0.000000, 0.167182]),
    np.array([0.096074, 0.000000, 0.581701, 0.000000, 0.999999, 0.383890, 0.202189, 0.999999])
]
R3_X = [
    np.array([0.250000, 0.250000]),
    np.array([0.695000, 0.396000]),
    np.array([0.300000, 0.500000, 0.700000]),
    np.array([0.440000, 0.425000, 0.378000, 0.397000]),
    np.array([0.000000, 0.850000, 0.999999, 0.999999]),
    np.array([0.500000, 0.500000, 0.500000, 0.500000, 0.500000]),
    np.array([0.000000, 0.242000, 0.328000, 0.218000, 0.375000, 0.748000]),
    np.array([0.064000, 0.008000, 0.120000, 0.000000, 0.999999, 0.382000, 0.031000, 0.806000])
]
R4_X = [
    np.array([0.500000, 0.500000]),
    np.array([0.700000, 0.200000]),
    np.array([0.950000, 0.010000, 0.990000]),
    np.array([0.999999, 0.000000, 0.000000, 0.700000]),
    np.array([0.000000, 0.000000, 0.500000, 0.500000]),
    np.array([0.300000, 0.400000, 0.600000, 0.200000, 0.600000]),
    np.array([0.000000, 0.150000, 0.000000, 0.100000, 0.000000, 0.100000]),
    np.array([0.100000, 0.000000, 0.800000, 0.000000, 0.999999, 0.380000, 0.350000, 0.999999])
]

R1_Y = [-2.4675e-270, 0.7237404632835625, -0.08911956876452833,
         0.25957575200735095,  2105.928152398213,  -0.5507747202906804,
         2.207308607344047,    9.8595486103895]
R2_Y = [ 1.5176e-192,  0.5297658866453171, -0.23982430098711077,
        -27.859767965401783,   1616.625747348229,  -1.0045153236844038,
         0.050978228653516464,  9.2933769573024]
R3_Y = [ 9.7977e-42,   0.5263661301012157, -0.1139602029925284,
          0.2748080020297299,   2932.694991178572,  -1.0159268487405835,
          2.2071746109147172,   9.8591545999995]
R4_Y = [ 2.6753e-9,    0.5813540452269076, -0.4594065810473597,
        -30.894440825162423,   83.9625,           -1.223884840915805,
          0.02363347322274405,  8.5129002799994]

# Combine all rounds per function
ALL_X = [np.vstack([R1_X[i], R2_X[i], R3_X[i], R4_X[i]]) for i in range(8)]
ALL_Y = [np.array([R1_Y[i], R2_Y[i], R3_Y[i], R4_Y[i]]) for i in range(8)]
DIMS  = [x.shape[1] for x in ALL_X]

print('Function dimensions:', DIMS)
print('\nPer-function best Y so far (maximisation objective):')
rows = []
for i, ys in enumerate(ALL_Y):
    bi = np.argmax(ys)
    rounds = ['R1','R2','R3','R4']
    rows.append({'f': f'f{i+1}', 'dim': DIMS[i],
                 'R1': f'{R1_Y[i]:.4g}', 'R2': f'{R2_Y[i]:.4g}',
                 'R3': f'{R3_Y[i]:.4g}', 'R4': f'{R4_Y[i]:.4g}',
                 'Best Round': rounds[bi], 'Best Y': f'{ys[bi]:.4g}'})
df = pd.DataFrame(rows)
print(df.to_string(index=False))

In [ ]:
# ── GP SURROGATE: exact R4 architecture ─────────────────────────────────────
# Key correction vs Round 4: acquisition uses MAXIMISATION EI
# R4 error: y_best = np.min(ALL_Y[i])  ← minimisation convention
# R5 fix:   y_best = np.max(ALL_Y[i])  ← correct maximisation convention

def build_gp(X, y):
    """Fit GP with Matérn-5/2 ARD + WhiteKernel. StandardScaler on X."""
    sc = StandardScaler()
    Xs = sc.fit_transform(X)
    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(length_scale=np.ones(X.shape[1]),
                 length_scale_bounds=(1e-2, 10.0), nu=2.5)
        + WhiteKernel(noise_level=1e-4, noise_level_bounds=(1e-8, 1e-1))
    )
    gpr = GaussianProcessRegressor(
        kernel=kernel, n_restarts_optimizer=10,
        normalize_y=True, random_state=42
    )
    gpr.fit(Xs, y)
    return gpr, sc


def ei_maximisation(X_cand, gpr, sc, y_best, xi=0.01):
    """
    Expected Improvement for MAXIMISATION.
    EI = (mu - y_best - xi) * Phi(Z) + sigma * phi(Z)
    Z  = (mu - y_best - xi) / sigma
    """
    Xs = sc.transform(X_cand)
    mu, sigma = gpr.predict(Xs, return_std=True)
    sigma = np.maximum(sigma, 1e-9)
    Z  = (mu - y_best - xi) / sigma
    ei = (mu - y_best - xi) * norm.cdf(Z) + sigma * norm.pdf(Z)
    ei[ei < 0] = 0.0
    return ei


def optimise_ei(func_idx, gpr, sc, y_best, xi, bounds_override=None, n_restarts=35):
    """Multi-start L-BFGS-B maximisation of EI."""
    d      = DIMS[func_idx]
    bounds = bounds_override or [(0.0, 0.999999)] * d
    best_x = ALL_X[func_idx][np.argmax(ALL_Y[func_idx])]
    np.random.seed(42)
    starts = [
        np.clip(best_x + np.random.randn(d) * 0.05,
                [b[0] for b in bounds], [b[1] for b in bounds])
        for _ in range(n_restarts - 1)
    ]
    starts.append(np.clip(best_x, [b[0] for b in bounds], [b[1] for b in bounds]))
    best_val, best_candidate = -np.inf, None
    for x0 in starts:
        res = minimize(
            lambda x: -ei_maximisation(x.reshape(1, -1), gpr, sc, y_best, xi).item(),
            x0, bounds=bounds, method='L-BFGS-B',
            options={'maxiter': 200, 'ftol': 1e-9}
        )
        if -res.fun > best_val:
            best_val = -res.fun
            best_candidate = np.clip(res.x, [b[0] for b in bounds], [b[1] for b in bounds])
    return best_candidate, best_val


# Fit GP for all 8 functions
GP_MODELS  = []
GP_SCALERS = []
for i in range(8):
    gpr, sc = build_gp(ALL_X[i], ALL_Y[i])
    GP_MODELS.append(gpr)
    GP_SCALERS.append(sc)
    print(f'f{i+1}: {gpr.kernel_}  | LML={gpr.log_marginal_likelihood_value_:.3f}')

In [ ]:
# ── NN SURROGATE: gradient/sensitivity analysis ──────────────────────────────

def build_nn_surrogate(X, y, hidden=(32, 32), max_iter=2000):
    sc = StandardScaler()
    Xs = sc.fit_transform(X)
    nn = MLPRegressor(
        hidden_layer_sizes=hidden, activation='relu',
        solver='lbfgs', alpha=1e-3, max_iter=max_iter, random_state=42
    )
    nn.fit(Xs, y)
    return nn, sc


def nn_gradient_fd(nn, sc, x0, eps=1e-4):
    """Central finite-difference gradient — backpropagation proxy."""
    x0   = np.array(x0, dtype=float)
    grad = np.zeros_like(x0)
    for j in range(len(x0)):
        xp = x0.copy(); xp[j] += eps
        xm = x0.copy(); xm[j] -= eps
        fp = nn.predict(sc.transform(xp.reshape(1, -1)))[0]
        fm = nn.predict(sc.transform(xm.reshape(1, -1)))[0]
        grad[j] = (fp - fm) / (2 * eps)
    return grad


print('Training NN surrogate for each function...\n')
NN_MODELS    = []
NN_SCALERS   = []
NN_GRADIENTS = []

for i in range(8):
    nn, sc = build_nn_surrogate(ALL_X[i], ALL_Y[i])
    NN_MODELS.append(nn)
    NN_SCALERS.append(sc)
    best_x = ALL_X[i][np.argmax(ALL_Y[i])]
    grad   = nn_gradient_fd(nn, sc, best_x)
    NN_GRADIENTS.append(grad)
    print(f'f{i+1} | best_x = {np.round(best_x, 4)}')
    print(f'      | NN gradient ∂ŷ/∂xᵢ = {np.round(grad, 4)}')
    print(f'      | Top sensitivity dim: x{np.argmax(np.abs(grad))+1}  (grad={grad[np.argmax(np.abs(grad))]:.4f})')
    print()

In [ ]:
# ── GRADIENT MAGNITUDE CHART ─────────────────────────────────────────────────

fig, axes = plt.subplots(2, 4, figsize=(18, 7))
fig.suptitle(
    'NN Surrogate — Gradient Magnitudes at Best Known Point (R1–R4)\n'
    '|∂ŷ/∂xᵢ| — backpropagation sensitivity proxy',
    fontsize=13, fontweight='bold'
)
for i, ax in enumerate(axes.flat):
    grad   = np.abs(NN_GRADIENTS[i])
    d      = DIMS[i]
    colors = ['#e74c3c' if g == grad.max() else '#3498db' for g in grad]
    ax.bar([f'x{j+1}' for j in range(d)], grad,
           color=colors, edgecolor='white', linewidth=0.8)
    ax.set_title(f'f{i+1}  (dim={d})', fontweight='bold', fontsize=11)
    ax.set_ylabel('|∂ŷ/∂xᵢ|', fontsize=9)
    ax.set_xlabel('Dimension', fontsize=9)
    max_j = int(np.argmax(grad))
    if grad[max_j] > 0:
        ax.annotate(f'max\nx{max_j+1}', xy=(max_j, grad[max_j]),
                    xytext=(max_j, grad[max_j]*1.12), ha='center',
                    fontsize=8, color='#c0392b', fontweight='bold')
    ax.set_ylim(0, grad.max() * 1.35 if grad.max() > 0 else 1)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig('gradient_heatmap_r5.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gradient heatmap saved.')

In [ ]:
# ── ROUND 5 QUERY STRATEGY ───────────────────────────────────────────────────
#
# Decision logic (same three-layer hierarchy as R4, with objective correction):
#   1. GP-EI (maximisation) proposes the baseline candidate.
#   2. NN gradient at the best-known point informs directional bias.
#   3. Domain knowledge overrides where GP posterior is unreliable
#      (sigma > 5x best_Y range, or L-BFGS-B hits constraint boundary).
#
# PER-FUNCTION CONSTRAINTS
# f4: interior safe zone (0.25–0.65)^4 — boundary x1=1 causes −28/−30 collapse
# f5: x1 locked [0, 0.0001]; x2 NN-dominant (grad=4751) → push to boundary;
#     x3/x4 boundary confirmed by R1/R3
# f6: x4 high [0.8, 1.0] + x5 low [0, 0.05] — structural pattern from R1 best
# f7: x1 locked [0, 0.001]; tight neighbourhood of R1/R3 confirmed optimum
# f8: tight neighbourhood of R1/R3 best; x5 boundary [0.999, 0.999999] locked

STRATEGIES = [
    # idx  xi      bounds                                            note
    {'xi': 1e-10, 'b': [(0., 0.999999)] * 2,
     'note': 'f1: flat landscape; GP-EI free exploration'},
    {'xi': 0.05,  'b': [(0., 0.999999)] * 2,
     'note': 'f2: exploit near R1 best (0.695, 0.396) = 0.724'},
    {'xi': 0.005, 'b': [(0., 0.999999)] * 3,
     'note': 'f3: exploit near R1 best; avoid corner boundaries'},
    {'xi': 0.01,  'b': [(0.25, 0.65)] * 4,
     'note': 'f4: interior-constrained; boundary causes −28/−30 collapse'},
    {'xi': 50.0,  'b': [(0., 0.0001), (0.85, 0.999999), (0.999, 0.999999), (0.999, 0.999999)],
     'note': 'f5: x1=0 locked; NN x2-dominant (grad=4751); push x2→1'},
    {'xi': 0.02,  'b': [(0., 0.999999), (0., 0.999999), (0., 0.999999), (0.8, 0.999999), (0., 0.05)],
     'note': 'f6: x4 high + x5 low structural pattern; explore x1–x3'},
    {'xi': 0.001, 'b': [(0., 0.001), (0.20, 0.38), (0.25, 0.42), (0.15, 0.30), (0.30, 0.46), (0.70, 0.80)],
     'note': 'f7: x1=0 locked; tight exploit R1/R3 confirmed optimum'},
    {'xi': 0.005, 'b': [(0.04, 0.09), (0., 0.02), (0.09, 0.16), (0., 0.005), (0.999, 0.999999), (0.35, 0.42), (0.02, 0.055), (0.78, 0.83)],
     'note': 'f8: tight exploit R1/R3 best; x5 boundary locked at 0.999999'},
]

print('Running GP-EI optimisation (maximisation)...\n')
EI_QUERIES = []
for i in range(8):
    s     = STRATEGIES[i]
    y_best = np.max(ALL_Y[i])   # ← MAXIMISATION (corrected from R4)
    x_ei, ei_val = optimise_ei(i, GP_MODELS[i], GP_SCALERS[i], y_best, s['xi'], s['b'])
    EI_QUERIES.append(x_ei)
    mu, sig = GP_MODELS[i].predict(GP_SCALERS[i].transform(x_ei.reshape(1, -1)), return_std=True)
    print(f'f{i+1} | best_Y={y_best:.4g} | GP mu={mu[0]:.4f} ± {sig[0]:.4f} | EI={ei_val:.4g}')
    print(f'      | {s["note"]}')
    print(f'      | GP-EI query: {np.round(x_ei, 6)}')
    print()

In [ ]:
# ── ROUND 5 FINAL QUERIES ────────────────────────────────────────────────────
# GP-EI results accepted for f1, f2, f3, f5, f6, f8.
# Manual overrides for f4 and f7 where GP posterior is unreliable
# (sigma >> best_Y range) or NN gradient provides stronger directional signal.

ROUND5_QUERIES = [
    # f1 (2D): flat landscape; GP-EI free exploration near centre
    EI_QUERIES[0],

    # f2 (2D): GP reproduces R1 best (0.695, 0.396); tight exploitation
    EI_QUERIES[1],

    # f3 (3D): GP exploits near R1 best with slight x2/x3 shift
    EI_QUERIES[2],

    # f4 (4D): GP sigma=6.17 unreliable; L-BFGS-B hit constraint corner.
    #          Manual perturbation of confirmed interior best (0.44, 0.425, 0.378, 0.397).
    #          Shift: x1 +0.015, x2 −0.010, x3 +0.007, x4 −0.002.
    np.array([0.455000, 0.415000, 0.385000, 0.395000]),

    # f5 (4D): GP-EI + NN gradient (∂ŷ/∂x2 = 4751) → push x2 to 0.999999.
    #          x1=0 locked; x3/x4 boundary confirmed R1/R3.
    EI_QUERIES[4],

    # f6 (5D): GP-EI maintains x4=0.999999, x5=0; x1 pushed to 0.759 (unexplored).
    EI_QUERIES[5],

    # f7 (6D): R1 and R3 near-identical at 2.207. NN gradient: x4=+3.17, x3=+1.64, x5=+1.43.
    #          Manual nudge in positive-gradient direction from confirmed best.
    #          Base: [0., 0.242, 0.328, 0.218, 0.375, 0.748] → small positive shift on x2,x4,x5.
    np.array([0.000000, 0.260000, 0.340000, 0.232000, 0.395000, 0.752000]),

    # f8 (8D): GP tight exploitation (sigma=0.41); x5 boundary locked at 0.999999.
    EI_QUERIES[7],
]

# Flag which queries used GP-EI vs manual
SOURCES = ['GP-EI', 'GP-EI', 'GP-EI', 'Manual', 'GP-EI+NN', 'GP-EI', 'Manual+NN', 'GP-EI']

print('=' * 70)
print('ROUND 5 — FINAL QUERY POINTS')
print('Format: x1–x2–…–xn  (6 decimal places)')
print('=' * 70)

submission_rows = []
for i, q in enumerate(ROUND5_QUERIES):
    formatted = '–'.join([f'{v:.6f}' for v in q])
    best_y    = np.max(ALL_Y[i])
    best_r    = ['R1','R2','R3','R4'][np.argmax(ALL_Y[i])]
    top_dim   = int(np.argmax(np.abs(NN_GRADIENTS[i])))
    submission_rows.append({
        'f': f'f{i+1}', 'dim': DIMS[i],
        'best_Y': f'{best_y:.4g}', 'best_rnd': best_r,
        'NN top dim': f'x{top_dim+1}', 'source': SOURCES[i],
        'R5 Query': formatted
    })
sub_df = pd.DataFrame(submission_rows)
print(sub_df.to_string(index=False))

print('\n' + '=' * 70)
print('COPY-PASTE SUBMISSION LINES:')
print('=' * 70)
for i, q in enumerate(ROUND5_QUERIES):
    print(f'F{i+1}: ' + '-'.join([f'{v:.6f}' for v in q]))

In [ ]:
# ── GP POSTERIOR SLICE PLOTS ─────────────────────────────────────────────────

def gp_slice_plot(ax, func_idx, dim_to_vary, n_points=100):
    """GP posterior mean ± 2σ along one dimension, all others fixed at best_x."""
    X      = ALL_X[func_idx]
    y      = ALL_Y[func_idx]
    gpr    = GP_MODELS[func_idx]
    sc     = GP_SCALERS[func_idx]
    best_x = X[np.argmax(y)].copy()

    x_sweep = np.linspace(0, 1, n_points)
    X_cand  = np.tile(best_x, (n_points, 1))
    X_cand[:, dim_to_vary] = x_sweep
    mu, sigma = gpr.predict(sc.transform(X_cand), return_std=True)

    ax.plot(x_sweep, mu, 'b-', lw=2, label='GP mean')
    ax.fill_between(x_sweep, mu - 2*sigma, mu + 2*sigma,
                    alpha=0.25, color='steelblue', label='±2σ')
    rounds = ['R1','R2','R3','R4']
    for j, (xo, yo) in enumerate(zip(X, y)):
        ax.scatter(xo[dim_to_vary], yo, color='red', zorder=5, s=60,
                   label='observations' if j == 0 else '')
    q5_val = ROUND5_QUERIES[func_idx][dim_to_vary]
    ax.axvline(q5_val, color='green', ls='--', lw=1.5, label='R5 query')
    ax.set_title(f'f{func_idx+1} — slice x{dim_to_vary+1}', fontsize=10, fontweight='bold')
    ax.set_xlabel(f'x{dim_to_vary+1}', fontsize=9)
    ax.set_ylabel('f(x)', fontsize=9)


fig, axes = plt.subplots(2, 4, figsize=(20, 8))
fig.suptitle(
    'GP Posterior — 1-D Slice through Best Known Point  |  Most NN-sensitive dimension\n'
    'Blue = mean ± 2σ  |  Red = R1–R4 observations  |  Green = Round 5 query',
    fontsize=12, fontweight='bold'
)
for i, ax in enumerate(axes.flat):
    top_dim = int(np.argmax(np.abs(NN_GRADIENTS[i])))
    gp_slice_plot(ax, i, top_dim)
    ax.legend(fontsize=7, loc='best')
plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.savefig('gp_posterior_slices_r5.png', dpi=150, bbox_inches='tight')
plt.show()
print('GP posterior slices saved.')

In [ ]:
# ── R4 DIAGNOSTIC: OBJECTIVE INVERSION ───────────────────────────────────────
# Round 4 notebook used y_best = np.min(ALL_Y[i]) with minimisation EI.
# The portal objective is MAXIMISATION. This produced four deteriorations:

diagnostic = [
    {'f':'f4', 'R4_query':'[1,0,0,0.7]',    'R3_best':'+0.275 (interior)',   'R4_result':'-30.894', 'cause':'Exploited minimum (−27.86 boundary) instead of maximum'},
    {'f':'f5', 'R4_query':'[0,0,0.5,0.5]',  'R3_best':'+2932 (R3)',          'R4_result':'83.963',  'cause':'Treated 1616 (min of positives) as best; reduced x3/x4'},
    {'f':'f7', 'R4_query':'[0,0.15,0,0.1,0,0.1]', 'R3_best':'+2.207 (R1/R3)','R4_result':'0.024', 'cause':'Pushed toward min (0.051); reduced non-zero dims'},
    {'f':'f8', 'R4_query':'[0.1,0,0.8,0,1,0.38,0.35,1]','R3_best':'+9.859 (R1)','R4_result':'8.513','cause':'Exploited 9.293 (observed minimum); wrong gradient direction'},
]
print('R4 OBJECTIVE INVERSION DIAGNOSTIC')
print('='*80)
for d in diagnostic:
    print(f"{d['f']} | Best should have been {d['R3_best']}")
    print(f"     | R4 query: {d['R4_query']}  →  Result: {d['R4_result']}")
    print(f"     | Cause: {d['cause']}")
    print()
print('R5 correction: y_best = np.max(ALL_Y[i]) throughout.')